In [3]:
import os
os.environ["WANDB_START_METHOD"] = "thread"

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import wandb
import optuna
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [5]:
WANDB_ENABLED   = True
RUN_MODELS      = {
    'logistic_regression': True,
    'random_forest':       True,
    'gradient_boosting':   True,
    'xgboost':             True,
}
N_OPTUNA_TRIALS = 50
RANDOM_STATE    = 42

In [6]:
df_raw = pd.read_csv(
    r"C:\Users\17063\OneDrive\Documents\GitHub\DS_Capstone_Group_1\data\v2_cleaned\all_merged.csv"
)

def add_features(df):
    df = df.copy()
    df['providers_per_poverty_pop'] = df['pcp_count'] / (df['total_population_poverty'] + 1)
    df['log_total_population']      = np.log1p(df['total_population'])
    df['log_pcp_count']             = np.log1p(df['pcp_count'])
    return df

df_raw['hpsa_high_need'] = (df_raw['hpsa_score_max'] >= 14).astype(int)
df = add_features(df_raw)

designated = df[df['hpsa_designated'] == 1]
counts = designated['hpsa_high_need'].value_counts().sort_index()
total  = counts.sum()
print("Class balance in HPSA-designated tracts:")
print(f"  not_high_need (0): {counts[0]:,}  ({counts[0]/total*100:.1f}%)")
print(f"  high_need     (1): {counts[1]:,}  ({counts[1]/total*100:.1f}%)")
print(f"  imbalance ratio  : 1:{counts[1]/counts[0]:.1f}")

Class balance in HPSA-designated tracts:
  not_high_need (0): 7,537  (11.5%)
  high_need     (1): 58,086  (88.5%)
  imbalance ratio  : 1:7.7


In [7]:
FEATURES = [
    'uninsured_pct',
    'no_checkup_pct',
    'median_household_income',
    'poverty_rate_pct',
    'pcp_per_100k',
    'providers_per_poverty_pop',
    'log_total_population',
    'log_pcp_count',
]
TARGET      = 'hpsa_high_need'
CLASS_NAMES = ['not_high_need', 'high_need']

CATEGORY_LABELS = {
    0: 'hpsa_designated_low_need',
    1: 'hpsa_designated_high_need',
    2: 'non_hpsa_low_need',
    3: 'non_hpsa_high_need',
}

In [8]:
train_df   = df[df['hpsa_designated'] == 1].dropna(subset=FEATURES + [TARGET])
impute_df  = df[df['hpsa_designated'] == 0].dropna(subset=FEATURES).copy()

X      = train_df[FEATURES]
y      = train_df[TARGET]
groups = train_df['geoid']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training tracts : {len(X_train):,}  ({groups.iloc[train_idx].nunique()} counties)")
print(f"Test tracts     : {len(X_test):,}  ({groups.iloc[test_idx].nunique()} counties)")
print(f"Imputation rows : {len(impute_df):,}")

# scale_pos_weight for XGBoost (ratio of majority to minority)
vc = y_train.value_counts()
SCALE_POS_WEIGHT = float(vc[0] / vc[1])
print(f"\nSCALE_POS_WEIGHT (not_high_need / high_need): {SCALE_POS_WEIGHT:.3f}")

# per-sample weights for GradientBoosting (balanced)
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train.values
)
weight_map     = {0: class_weights_arr[0], 1: class_weights_arr[1]}
sample_weights = y_train.map(weight_map).values
print(f"Sample weight for class 0: {class_weights_arr[0]:.4f}")
print(f"Sample weight for class 1: {class_weights_arr[1]:.4f}")

Training tracts : 55,254  (2191 counties)
Test tracts     : 10,369  (548 counties)
Imputation rows : 5,883

SCALE_POS_WEIGHT (not_high_need / high_need): 0.123
Sample weight for class 0: 4.5574
Sample weight for class 1: 0.5616


In [9]:
all_metrics    = {}
trained_models = {}

def evaluate_model(name, pipeline, config=None, fit_params=None):
    pipeline.fit(X_train, y_train, **(fit_params or {}))
    preds      = pipeline.predict(X_test)
    pred_proba = pipeline.predict_proba(X_test)[:, 1]
    metrics = {
        'accuracy':    accuracy_score(y_test, preds),
        'weighted_f1': f1_score(y_test, preds, average='weighted'),
        'roc_auc':     roc_auc_score(y_test, pred_proba),
    }
    print(f"\n{'='*50}\n{name}")
    print(classification_report(y_test, preds, target_names=CLASS_NAMES))
    print(f"ROC-AUC : {metrics['roc_auc']:.4f}")

    if WANDB_ENABLED:
        run = wandb.init(
            project='hpsa-classification',
            name=name,
            config=config or {},
            reinit=True
        )
        wandb.log(metrics)
        y_probas = np.column_stack([1 - pred_proba, pred_proba])
        wandb.log({'roc_curve': wandb.plot.roc_curve(y_test, y_probas, labels=CLASS_NAMES,
                                                      title=f'ROC — {name}')})
        run.finish()

    all_metrics[name]    = metrics
    trained_models[name] = pipeline
    return pipeline

In [10]:
if RUN_MODELS['logistic_regression']:
    lr_param_grid = {
        'model__C':       [0.001, 0.01, 0.1, 1, 10, 100],
        'model__penalty': ['l1', 'l2'],
    }
    lr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(
            class_weight='balanced',
            solver='liblinear',
            random_state=RANDOM_STATE,
            max_iter=1000
        ))
    ])
    lr_gs = GridSearchCV(
        lr_pipe,
        lr_param_grid,
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )
    lr_gs.fit(X_train, y_train)
    print(f"LR best params : {lr_gs.best_params_}")
    print(f"LR best CV AUC : {lr_gs.best_score_:.4f}")
    evaluate_model(
        'logistic_regression_v3_gridsearch',
        lr_gs.best_estimator_,
        config={**lr_gs.best_params_, 'model_type': 'LogisticRegression',
                'class_weight': 'balanced', 'cv': 'StratifiedKFold(5)'}
    )

Fitting 5 folds for each of 12 candidates, totalling 60 fits
LR best params : {'model__C': 100, 'model__penalty': 'l2'}
LR best CV AUC : 0.8259

logistic_regression_v3_gridsearch
               precision    recall  f1-score   support

not_high_need       0.22      0.68      0.34      1475
    high_need       0.92      0.61      0.73      8894

     accuracy                           0.62     10369
    macro avg       0.57      0.65      0.54     10369
 weighted avg       0.82      0.62      0.68     10369

ROC-AUC : 0.7268


Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ctrazona1385 (mmcgee18-georgia-state-university). Use `wandb login --relogin` to force relogin


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.62002
roc_auc,0.72682
weighted_f1,0.67727


In [11]:
if RUN_MODELS['random_forest']:
    rf_param_grid = {
        'n_estimators':     [100, 200, 300],
        'max_depth':        [None, 10, 20, 30],
        'min_samples_split':[2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features':     ['sqrt', 'log2'],
    }
    rf_gs = GridSearchCV(
        RandomForestClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        rf_param_grid,
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )
    rf_gs.fit(X_train, y_train)
    print(f"RF best params : {rf_gs.best_params_}")
    print(f"RF best CV AUC : {rf_gs.best_score_:.4f}")
    evaluate_model(
        'random_forest_v3_gridsearch',
        rf_gs.best_estimator_,
        config={**rf_gs.best_params_, 'model_type': 'RandomForestClassifier',
                'class_weight': 'balanced', 'cv': 'StratifiedKFold(5)'}
    )

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
RF best params : {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
RF best CV AUC : 0.9999

random_forest_v3_gridsearch
               precision    recall  f1-score   support

not_high_need       0.38      0.06      0.11      1475
    high_need       0.86      0.98      0.92      8894

     accuracy                           0.85     10369
    macro avg       0.62      0.52      0.51     10369
 weighted avg       0.80      0.85      0.80     10369

ROC-AUC : 0.6564


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.85206
roc_auc,0.65642
weighted_f1,0.80423


In [12]:
if RUN_MODELS['gradient_boosting']:
    skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

    def gb_objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 50, 400),
            'max_depth':         trial.suggest_int('max_depth', 2, 8),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        }
        fold_aucs = []
        X_arr = X_train.values
        y_arr = y_train.values
        for tr_idx, val_idx in skf.split(X_arr, y_arr):
            X_tr,  X_val  = X_arr[tr_idx],  X_arr[val_idx]
            y_tr,  y_val  = y_arr[tr_idx],  y_arr[val_idx]
            sw_tr         = sample_weights[tr_idx]

            scaler = StandardScaler().fit(X_tr)
            X_tr   = scaler.transform(X_tr)
            X_val  = scaler.transform(X_val)

            clf = GradientBoostingClassifier(**params, random_state=RANDOM_STATE)
            clf.fit(X_tr, y_tr, sample_weight=sw_tr)
            fold_aucs.append(roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1]))
        return np.mean(fold_aucs)

    gb_study = optuna.create_study(direction='maximize', study_name='gb_v3')
    gb_study.optimize(gb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

    print(f"GB best CV AUC : {gb_study.best_value:.4f}")
    print(f"GB best params : {gb_study.best_params}")

    if WANDB_ENABLED:
        trial_run = wandb.init(
            project='hpsa-classification',
            name='gradient_boosting_v3_optuna_trials',
            reinit=True
        )
        trial_data = [
            [t.number, t.value, *t.params.values()]
            for t in gb_study.trials if t.value is not None
        ]
        cols = ['trial', 'roc_auc'] + list(gb_study.best_params.keys())
        wandb.log({'optuna_trials': wandb.Table(columns=cols, data=trial_data)})
        trial_run.finish()

    gb_best = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  GradientBoostingClassifier(**gb_study.best_params, random_state=RANDOM_STATE))
    ])
    evaluate_model(
        'gradient_boosting_v3_optuna',
        gb_best,
        config={**gb_study.best_params, 'model_type': 'GradientBoostingClassifier',
                'imbalance': 'sample_weight_balanced', 'cv': 'StratifiedKFold(5)_manual'},
        fit_params={'model__sample_weight': sample_weights}
    )

  0%|          | 0/50 [00:00<?, ?it/s]

GB best CV AUC : 0.9996
GB best params : {'n_estimators': 258, 'max_depth': 7, 'learning_rate': 0.07421544743597963, 'subsample': 0.6495490714541374, 'min_samples_split': 10, 'min_samples_leaf': 9, 'max_features': 'sqrt'}



gradient_boosting_v3_optuna
               precision    recall  f1-score   support

not_high_need       0.33      0.20      0.25      1475
    high_need       0.88      0.93      0.90      8894

     accuracy                           0.83     10369
    macro avg       0.60      0.57      0.58     10369
 weighted avg       0.80      0.83      0.81     10369

ROC-AUC : 0.6607


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.82901
roc_auc,0.66071
weighted_f1,0.81075


In [13]:
if RUN_MODELS['xgboost']:

    def xgb_objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
            'max_depth':        trial.suggest_int('max_depth', 2, 10),
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('model',  XGBClassifier(
                **params,
                scale_pos_weight=SCALE_POS_WEIGHT,
                random_state=RANDOM_STATE,
                eval_metric='logloss',
                verbosity=0,
                use_label_encoder=False
            ))
        ])
        scores = cross_val_score(
            clf, X_train, y_train,
            cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
            scoring='roc_auc',
            n_jobs=-1
        )
        return scores.mean()

    xgb_study = optuna.create_study(direction='maximize', study_name='xgb_v3')
    xgb_study.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

    print(f"XGB best CV AUC : {xgb_study.best_value:.4f}")
    print(f"XGB best params : {xgb_study.best_params}")

    if WANDB_ENABLED:
        trial_run = wandb.init(
            project='hpsa-classification',
            name='xgboost_v3_optuna_trials',
            reinit=True
        )
        trial_data = [
            [t.number, t.value, *t.params.values()]
            for t in xgb_study.trials if t.value is not None
        ]
        cols = ['trial', 'roc_auc'] + list(xgb_study.best_params.keys())
        wandb.log({'optuna_trials': wandb.Table(columns=cols, data=trial_data)})
        trial_run.finish()

    xgb_best = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  XGBClassifier(
            **xgb_study.best_params,
            scale_pos_weight=SCALE_POS_WEIGHT,
            random_state=RANDOM_STATE,
            eval_metric='logloss',
            verbosity=0,
            use_label_encoder=False
        ))
    ])
    evaluate_model(
        'xgboost_v3_optuna',
        xgb_best,
        config={**xgb_study.best_params, 'model_type': 'XGBClassifier',
                'scale_pos_weight': SCALE_POS_WEIGHT, 'cv': 'StratifiedKFold(5)'}
    )

  0%|          | 0/50 [00:00<?, ?it/s]

XGB best CV AUC : 0.9996
XGB best params : {'n_estimators': 495, 'max_depth': 10, 'learning_rate': 0.2909576760717733, 'subsample': 0.9904583029412303, 'colsample_bytree': 0.8262801060921539, 'min_child_weight': 3, 'gamma': 0.013713261955734005, 'reg_alpha': 0.962058314656758, 'reg_lambda': 5.3023501553627446e-08}



xgboost_v3_optuna
               precision    recall  f1-score   support

not_high_need       0.28      0.18      0.22      1475
    high_need       0.87      0.92      0.90      8894

     accuracy                           0.82     10369
    macro avg       0.57      0.55      0.56     10369
 weighted avg       0.79      0.82      0.80     10369

ROC-AUC : 0.6333


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.81628
roc_auc,0.63326
weighted_f1,0.7996


In [14]:
# SMOTE variants for all 4 models using best hyperparams from above

smote = SMOTE(random_state=RANDOM_STATE)

# --- Logistic Regression + SMOTE ---
if RUN_MODELS['logistic_regression']:
    best_lr_params = lr_gs.best_params_  # e.g. {'model__C': 1, 'model__penalty': 'l2'}
    lr_smote_pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(
            C=best_lr_params.get('model__C', 1),
            penalty=best_lr_params.get('model__penalty', 'l2'),
            class_weight='balanced',
            solver='liblinear',
            random_state=RANDOM_STATE,
            max_iter=1000
        ))
    ])
    evaluate_model(
        'logistic_regression_v3_smote',
        lr_smote_pipe,
        config={**best_lr_params, 'model_type': 'LogisticRegression',
                'class_weight': 'balanced', 'imbalance': 'SMOTE'}
    )

# --- Random Forest + SMOTE ---
if RUN_MODELS['random_forest']:
    best_rf_params = rf_gs.best_params_
    rf_smote_pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('model',  RandomForestClassifier(
            **best_rf_params,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    evaluate_model(
        'random_forest_v3_smote',
        rf_smote_pipe,
        config={**best_rf_params, 'model_type': 'RandomForestClassifier',
                'class_weight': 'balanced', 'imbalance': 'SMOTE'}
    )

# --- Gradient Boosting + SMOTE ---
if RUN_MODELS['gradient_boosting']:
    gb_smote_pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('model',  GradientBoostingClassifier(
            **gb_study.best_params,
            random_state=RANDOM_STATE
        ))
    ])
    evaluate_model(
        'gradient_boosting_v3_smote',
        gb_smote_pipe,
        config={**gb_study.best_params, 'model_type': 'GradientBoostingClassifier',
                'imbalance': 'SMOTE'}
    )

# --- XGBoost + SMOTE (scale_pos_weight=1.0 — SMOTE already balanced) ---
if RUN_MODELS['xgboost']:
    xgb_smote_pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('model',  XGBClassifier(
            **xgb_study.best_params,
            scale_pos_weight=1.0,
            random_state=RANDOM_STATE,
            eval_metric='logloss',
            verbosity=0,
            use_label_encoder=False
        ))
    ])
    evaluate_model(
        'xgboost_v3_smote',
        xgb_smote_pipe,
        config={**xgb_study.best_params, 'model_type': 'XGBClassifier',
                'scale_pos_weight': 1.0, 'imbalance': 'SMOTE'}
    )


logistic_regression_v3_smote
               precision    recall  f1-score   support

not_high_need       0.22      0.68      0.34      1475
    high_need       0.92      0.61      0.73      8894

     accuracy                           0.62     10369
    macro avg       0.57      0.64      0.53     10369
 weighted avg       0.82      0.62      0.68     10369

ROC-AUC : 0.7264


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.61771
roc_auc,0.72636
weighted_f1,0.67531



random_forest_v3_smote
               precision    recall  f1-score   support

not_high_need       0.35      0.17      0.23      1475
    high_need       0.87      0.95      0.91      8894

     accuracy                           0.84     10369
    macro avg       0.61      0.56      0.57     10369
 weighted avg       0.80      0.84      0.81     10369

ROC-AUC : 0.6293


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.83605
roc_auc,0.62933
weighted_f1,0.81151



gradient_boosting_v3_smote
               precision    recall  f1-score   support

not_high_need       0.36      0.19      0.25      1475
    high_need       0.88      0.94      0.91      8894

     accuracy                           0.84     10369
    macro avg       0.62      0.57      0.58     10369
 weighted avg       0.80      0.84      0.82     10369

ROC-AUC : 0.6971


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.83692
roc_auc,0.69707
weighted_f1,0.81511



xgboost_v3_smote
               precision    recall  f1-score   support

not_high_need       0.29      0.23      0.26      1475
    high_need       0.88      0.91      0.89      8894

     accuracy                           0.81     10369
    macro avg       0.59      0.57      0.57     10369
 weighted avg       0.79      0.81      0.80     10369

ROC-AUC : 0.6715


accuracy,▁
roc_auc,▁
weighted_f1,▁
accuracy,0.81223
roc_auc,0.67149
weighted_f1,0.80208


In [15]:
metrics_df = pd.DataFrame(all_metrics).T.round(4).sort_values('roc_auc', ascending=False)
print("\nModel comparison (sorted by ROC-AUC):")
print(metrics_df.to_string())


Model comparison (sorted by ROC-AUC):
                                   accuracy  weighted_f1  roc_auc
logistic_regression_v3_gridsearch    0.6200       0.6773   0.7268
logistic_regression_v3_smote         0.6177       0.6753   0.7264
gradient_boosting_v3_smote           0.8369       0.8151   0.6971
xgboost_v3_smote                     0.8122       0.8021   0.6715
gradient_boosting_v3_optuna          0.8290       0.8107   0.6607
random_forest_v3_gridsearch          0.8521       0.8042   0.6564
xgboost_v3_optuna                    0.8163       0.7996   0.6333
random_forest_v3_smote               0.8360       0.8115   0.6293


In [16]:
best_model_name = metrics_df['roc_auc'].idxmax()
best_model      = trained_models[best_model_name]
print(f"Best model : {best_model_name}")
print(f"ROC-AUC    : {metrics_df.loc[best_model_name, 'roc_auc']:.4f}")

Best model : logistic_regression_v3_gridsearch
ROC-AUC    : 0.7268


In [17]:
X_impute = impute_df[FEATURES]
non_hpsa_preds = best_model.predict(X_impute)

# Build full output on all tracts
output = df[['geoid_tract', 'geoid', 'hpsa_designated', 'hpsa_high_need']].copy()

# Designated rows: use actual label (0→cat 0, 1→cat 1)
output.loc[output['hpsa_designated'] == 1, 'hpsa_need_category'] = (
    output.loc[output['hpsa_designated'] == 1, 'hpsa_high_need'].astype(int)
)

# Non-designated rows: use model prediction (0→cat 2, 1→cat 3)
non_mask = output['hpsa_designated'] == 0
output.loc[non_mask, 'hpsa_need_category'] = (
    pd.Series(non_hpsa_preds, index=impute_df.index).map({0: 2, 1: 3})
)

output['hpsa_need_category'] = output['hpsa_need_category'].astype(int)
output['hpsa_need_category_label'] = output['hpsa_need_category'].map(CATEGORY_LABELS)

print("4-category distribution:")
print(output['hpsa_need_category_label'].value_counts().to_string())
print(f"\nNull check: {output[['hpsa_need_category','hpsa_need_category_label']].isnull().sum().sum()} nulls")

4-category distribution:
hpsa_need_category_label
hpsa_designated_high_need    58086
hpsa_designated_low_need      7537
non_hpsa_low_need             5317
non_hpsa_high_need             566

Null check: 0 nulls


In [18]:
out_path = r"C:\Users\17063\OneDrive\Documents\GitHub\DS_Capstone_Group_1\data\v2_cleaned\hpsa_need_category_v3.csv"
export_df = output[['geoid_tract', 'geoid', 'hpsa_designated', 'hpsa_need_category', 'hpsa_need_category_label']]
export_df.to_csv(out_path, index=False)
print(f"Exported {len(export_df):,} rows → {out_path}")
print(f"Columns : {list(export_df.columns)}")
print(f"Nulls   : {export_df.isnull().sum().sum()}")
print(f"Best model used: {best_model_name}")

Exported 71,506 rows → C:\Users\17063\OneDrive\Documents\GitHub\DS_Capstone_Group_1\data\v2_cleaned\hpsa_need_category_v3.csv
Columns : ['geoid_tract', 'geoid', 'hpsa_designated', 'hpsa_need_category', 'hpsa_need_category_label']
Nulls   : 0
Best model used: logistic_regression_v3_gridsearch
